# Plan B schema-linker evaluation

This notebook is a thin inspection client for the production evaluator. Build the index and publish an evaluation report with `scripts/13_schema_linker.py`; do not initialize or train a second linker here.

In [ ]:
from __future__ import annotations

import json
import platform
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd

from nl2sparql.linking.schema.evaluate import evaluate_linker, load_ground_truth

environment = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "package": version("nl2sparql-blockchain-kg"),
}
environment

## Artifact status

Missing ground truth or model/cache evidence is a blocked external gate. A malformed catalog, cache, ground truth, or report is a failed validation gate.

In [ ]:
GROUND_TRUTH_PATH = Path("../data/dataset/test/schema_linker_ground_truth.jsonl")
REPORT_PATH = Path("../reports/schema_linker_evaluation.json")
CACHE_MANIFEST_PATH = Path("../src/nl2sparql/linking/cache/schema-index.json")
CACHE_MATRICES_PATH = Path("../src/nl2sparql/linking/cache/schema-index.npz")

artifact_status = pd.DataFrame(
    [
        {"artifact": name, "path": str(path), "exists": path.is_file()}
        for name, path in {
            "ground_truth": GROUND_TRUTH_PATH,
            "evaluation_report": REPORT_PATH,
            "cache_manifest": CACHE_MANIFEST_PATH,
            "cache_matrices": CACHE_MATRICES_PATH,
        }.items()
    ]
)
artifact_status

## Production evaluation seam

Pass a production `SchemaLinker` and its catalog elements into this helper only when rerunning metrics intentionally. The notebook neither creates an encoder nor writes an index.

In [ ]:
def run_production_evaluation(linker, valid_elements, ground_truth_path=GROUND_TRUTH_PATH):
    cases = load_ground_truth(ground_truth_path, valid_elements)
    return evaluate_linker(linker, cases, field_k=10, relation_k=5)

## Published metrics

In [ ]:
report = json.loads(REPORT_PATH.read_text(encoding="utf-8")) if REPORT_PATH.is_file() else None
metric_table = (
    pd.DataFrame([{"metric": key, "value": value} for key, value in report["metrics"].items()])
    if report is not None
    else pd.DataFrame(columns=["metric", "value"])
)
metric_table

## Top retrieval errors

In [ ]:
top_errors = (
    pd.DataFrame(report["results"])
    .sort_values(["field_reciprocal_rank", "field_hits", "id"])
    .head(10)
    if report is not None
    else pd.DataFrame()
)
top_errors

## Manual gate

Before accepting evidence, confirm all 50 questions were independently reviewed, the report status is `ready`, digests match the intended catalog/cache/ground truth, Recall@K and MRR are plausible, p50/p95 exclude cold-start warm-up, and the worst-ranked cases have been inspected. Record any missing external model/network/ground truth as **blocked** and any invalid contract/index/evaluation as **failed**.